# Análisis de Cavidades en RuBisCO

Pipeline: fpocket → freesasa → APBS → Python

Basado en Poudel et al. (2020) — *Biophysical Analysis of the Structural Evolution of Substrate Specificity in RuBisCO*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fpino73/analisismolecular/blob/master/notebooks/francisco/03_analisis_cavidades_rubisco.ipynb)

## 0. Instalación (ejecutar una sola vez)

In [ ]:
# 0a. Clonar repo
import os
if not os.path.exists("/content/analisismolecular"):
    !git clone https://github.com/fpino73/analisismolecular.git /content/analisismolecular
%cd /content/analisismolecular
!git pull

In [ ]:
# 0b. Dependencias Python
!pip install -q -e .
!pip install -q prody biopython freesasa pdb2pqr

In [ ]:
# 0c. Compilar fpocket desde fuente (no está en apt de Colab)
import os, subprocess
if not os.path.exists("/usr/local/bin/fpocket"):
    !apt-get install -y -qq build-essential > /dev/null 2>&1
    %cd /tmp
    !rm -rf fpocket-src
    !git clone --depth 1 https://github.com/Discngine/fpocket.git fpocket-src 2>&1 | tail -1
    %cd fpocket-src
    !sed -i 's/-m64//g' makefile  # Colab no acepta -m64
    !make -j4 > /dev/null 2>&1
    !cp bin/fpocket /usr/local/bin/
    %cd /content/analisismolecular
print("fpocket instalado:", subprocess.run(["which", "fpocket"], capture_output=True, text=True).stdout.strip())

In [ ]:
# 0d. APBS desde apt-get
!apt-get install -y -qq apbs > /dev/null 2>&1
!which apbs && echo "APBS OK"

---
## 1. Imports y configuración

In [ ]:
import sys
sys.path.insert(0, "/content/analisismolecular/src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import urllib.request
import subprocess
import json
import tempfile
import shutil

sns.set_style("whitegrid")
%matplotlib inline
print("Listo.")

## 2. Ejecutar fpocket sobre una PDB

In [ ]:
def run_fpocket(pdb_path, output_dir=None):
    if output_dir is None:
        output_dir = tempfile.mkdtemp(prefix="fpocket_")
    else:
        Path(output_dir).mkdir(parents=True, exist_ok=True)
    cmd = ["fpocket", "-f", str(pdb_path), "-o", output_dir]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"fpocket falló:\n{result.stderr}")
    return output_dir

def parse_fpocket_output(fpocket_dir):
    fp_dir = Path(fpocket_dir)
    info_files = sorted(fp_dir.glob("*_info.txt"))
    pockets = []
    for info_path in info_files:
        with open(info_path) as f:
            lines = f.readlines()
        data = {}
        for line in lines:
            if ":" in line:
                key, _, val = line.partition(":")
                key = key.strip().lower().replace(" ", "_")
                val = val.strip()
                try:
                    data[key] = float(val)
                except ValueError:
                    data[key] = val
        data["pocket_id"] = info_path.stem.replace("_info", "")
        pockets.append(data)
    return pd.DataFrame(pockets)

## 3. Descargar estructuras PDB

In [ ]:
PDB_IDS = {
    "4RUB": {"group": "G-I", "organism": "Nicotiana tabacum", "S": 77},
    "1GK8": {"group": "G-I", "organism": "Chlamydomonas reinhardtii", "S": 61},
    "1RBL": {"group": "G-II", "organism": "Rhodospirillum rubrum", "S": 13},
    "1GEH": {"group": "G-II", "organism": "Rhodospirillum rubrum", "S": 15},
    "1SV3": {"group": "G-III", "organism": "Archaeoglobus fulgidus", "S": 5},
}

PDB_DIR = Path("/content/pdb_files")
PDB_DIR.mkdir(exist_ok=True)

for pdb_id in PDB_IDS:
    pdb_path = PDB_DIR / f"{pdb_id}.pdb"
    if not pdb_path.exists():
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        print(f"Descargando {pdb_id}...")
        urllib.request.urlretrieve(url, pdb_path)

print("\nEstructuras listas:")
for pdb_id, info in PDB_IDS.items():
    print(f"  {pdb_id}  {info['group']:5s}  {info['organism']:30s}  S={info['S']}")

## 4. Pipeline: fpocket → análisis de cavidades

In [ ]:
ALL_RESULTS = []

for pdb_id, info in PDB_IDS.items():
    pdb_path = PDB_DIR / f"{pdb_id}.pdb"
    if not pdb_path.exists():
        continue

    print(f"\n--- {pdb_id} ({info['group']}) ---")

    # fpocket
    fp_dir = run_fpocket(str(pdb_path))
    pockets = parse_fpocket_output(fp_dir)
    print(f"  Cavidades encontradas: {len(pockets)}")

    if pockets.empty:
        continue

    # Anotar
    pockets["pdb_id"] = pdb_id
    pockets["group"] = info["group"]
    pockets["organism"] = info["organism"]
    pockets["S"] = info["S"]

    # Top 3 cavidades por score
    best = pockets.nlargest(3, "score") if "score" in pockets.columns else pockets.head(3)
    ALL_RESULTS.append(best)

df_all = pd.concat(ALL_RESULTS, ignore_index=True) if ALL_RESULTS else pd.DataFrame()
print(f"\nTotal cavidades top-3: {len(df_all)}")
df_all.head(10)

## 5. Visualización

In [ ]:
if not df_all.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Score vs S
    sns.scatterplot(data=df_all, x="score", y="S", hue="group", style="group", s=120, ax=axes[0])
    axes[0].set_xlabel("Score fpocket")
    axes[0].set_ylabel("S (CO₂/O₂)")
    axes[0].set_title("Score fpocket vs Especificidad")

    # Druggability score vs S
    if "druggability_score" in df_all.columns:
        sns.scatterplot(data=df_all, x="druggability_score", y="S", hue="group", style="group", s=120, ax=axes[1])
        axes[1].set_xlabel("Druggability Score")
        axes[1].set_ylabel("S (CO₂/O₂)")
        axes[1].set_title("Druggability vs Especificidad")
    else:
        axes[1].text(0.5, 0.5, "No disponible", ha="center", va="center", transform=axes[1].transAxes)

    # Conteo por grupo
    counts = df_all["group"].value_counts()
    counts.plot(kind="bar", ax=axes[2], color=["#2ecc71", "#3498db", "#e74c3c"])
    axes[2].set_xlabel("Grupo")
    axes[2].set_ylabel("N° cavidades (top-3)")
    axes[2].set_title("Cavidades por grupo")

    plt.tight_layout()
    plt.show()

In [ ]:
# Resumen por grupo
if not df_all.empty:
    cols = [c for c in ["score", "volume"] if c in df_all.columns]
    display(df_all.groupby("group")[cols + ["S"]].agg(["mean", "count"]).round(2))

## 6. Boxplot comparativo

In [ ]:
if not df_all.empty and "druggability_score" in df_all.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(data=df_all, x="group", y="druggability_score", palette="Set2", ax=ax)
    sns.stripplot(data=df_all, x="group", y="druggability_score", color="black", size=6, alpha=0.5, ax=ax)
    ax.set_xlabel("Grupo filogenético")
    ax.set_ylabel("Druggability Score")
    ax.set_title("Druggability de cavidades por grupo")
    plt.show()
else:
    print("No hay datos de druggability.")

## 7. Conclusiones

Discusión basada en Poudel 2020:

- **G-III no sigue la tendencia**: cavidades grandes pero S ~5 — el tamaño no lo explica todo
- **Cambio topológico**: G-I desarrolla un túnel continuo; G-II/G-III tienen bolsas aisladas
- **Electrostatic steering**: el gradiente catiónico en el túnel orienta CO₂ hacia el sitio activo
- **Próximo paso**: integrar APBS para filtrar cavidades por carga positiva y medir área exacta